# 01 Delta Lake Fundamentals

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Understand what Delta Lake is (Parquet files plus a transaction log), what it adds over plain Parquet (ACID transactions, schema enforcement, DML, time travel, file statistics), and the difference between <b>managed</b> and <b>external</b> tables in Unity Catalog.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Delta Lake is the storage layer of the Databricks lakehouse: every table you create on Databricks is a Delta table by default. The certification exam assumes you know how Delta provides reliability on cheap cloud storage, and "how does Delta provide ACID guarantees?" is one of the six spoken answers in this learning path.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Delta = Parquet + a transaction log</b><br>
<code>`</code><br>my_table/<br>├── _delta_log/<br>│   ├── 00000000000000000000.json      <- commit 0: "these files make up the table"<br>│   ├── 00000000000000000001.json      <- commit 1: "add file C, remove file A"<br>│   └── 00000000000000000010.checkpoint.parquet   <- periodic summary of the log<br>├── part-00000-....snappy.parquet      <- data files (plain Parquet)<br>└── part-00001-....snappy.parquet<br><code>`</code>
<ul><li>The <b>data</b> is ordinary Parquet files</li><li>The <b>transaction log</b> (<code>_delta_log</code>) is an ordered list of commits. Each commit records which files were <b>added</b> and <b>removed</b>, plus schema and metadata</li><li>A reader builds the table's current state by replaying the log, so it sees <b>only</b> files from committed versions. That's what makes writes atomic</li></ul>
</div>

| Feature | Plain Parquet folder | Delta table |
|---|---|---|
| Atomic writes | ❌ readers can see half-written data | ✅ commit or nothing |
| Concurrent writers | ❌ can corrupt the folder | ✅ optimistic concurrency control |
| Schema enforcement | ❌ any file can have any schema | ✅ rejects mismatched writes |
| `UPDATE` / `DELETE` / `MERGE` | ❌ rewrite everything yourself | ✅ built in |
| Time travel and rollback | ❌ | ✅ query or restore old versions |
| File statistics for data skipping | Row-group stats only | ✅ min/max per file in the log |
| Streaming source and sink | Limited | ✅ exactly-once |

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A bank's nightly job overwrites a Parquet folder of account balances. One night the job fails halfway: half the old files are deleted and half the new ones are written, and the morning reports show wrong totals. After moving to Delta, the same failure leaves the table at the previous version, because the new files were never committed. A bad but successful load can be undone with <code>RESTORE</code> in seconds.
</div>

## Managed vs external tables (Unity Catalog)

| | Managed table | External table |
|---|---|---|
| Created with | `CREATE TABLE t ...` / `saveAsTable("t")` (no location) | `CREATE TABLE t ... LOCATION 'abfss://...'` |
| Data stored in | Storage managed by Unity Catalog | A path you choose (an external location) |
| `DROP TABLE` | Deletes metadata **and** data (data is kept briefly for recovery with `UNDROP`) | Deletes metadata only. Files remain |
| Optimizations | Predictive optimization, automatic maintenance | You manage them |
| Recommended for | Almost everything | Data that other systems must read or write in place |

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
In Free Edition you create <b>managed</b> tables in the <code>workspace</code> catalog. External tables need an external location and storage credential, which are account admin features (covered conceptually in <code>06_unity_catalog</code> and <code>09_azure_databricks</code>).
</div>

## Setup: schema and volume for the Delta Lake folder

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the schema <code>workspace.delta_lake</code> used by every lesson in this folder, and a volume for files.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The table prefix and the volume path.
</div>

In [0]:
CATALOG, SCHEMA = "workspace", "delta_lake"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.files")
DB = f"{CATALOG}.{SCHEMA}"
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/files/lesson01"
print("tables in", DB, "| files in", BASE)

## 1. Create a Delta table

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Saves a DataFrame as a managed table and inspects it with <code>DESCRIBE DETAIL</code>.<br><br>
<b>Why it matters</b><br>On Databricks you don't need <code>format("delta")</code>: Delta is the default. <code>DESCRIBE DETAIL</code> shows the format, number of files, size, location and the Delta <b>protocol</b> versions (which features readers and writers must support).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>format = delta</code>, a few files, <code>minReaderVersion</code> and <code>minWriterVersion</code>, and a location managed by the catalog.
</div>

In [0]:
from pyspark.sql import functions as F

accounts = spark.createDataFrame(
    [(1, "Amal", 1200.0), (2, "John", 560.0), (3, "Vivek", 3100.0), (4, "Sara", 75.0)],
    "account_id INT, owner STRING, balance DOUBLE",
)
table = f"{DB}.accounts"
accounts.write.mode("overwrite").saveAsTable(table)

display(spark.sql(f"DESCRIBE DETAIL {table}").select(
    "format", "numFiles", "sizeInBytes", "minReaderVersion", "minWriterVersion", "location"))

## 2. Every write is a version

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Makes three changes (append, update, delete) and lists the table history.<br><br>
<b>Why it matters</b><br>Each successful operation is one <b>commit</b>, which is one <b>version</b> in the log. The history records who did what, when, with which parameters and metrics. That's the basis of auditing, time travel and rollback.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Versions 0 to 3: <code>CREATE OR REPLACE TABLE AS SELECT</code> (or <code>WRITE</code>), <code>WRITE</code> (append), <code>UPDATE</code> and <code>DELETE</code>, each with metrics such as rows written or files removed.
</div>

In [0]:
spark.createDataFrame([(5, "Priya", 900.0)], accounts.schema).write.mode("append").saveAsTable(table)
spark.sql(f"UPDATE {table} SET balance = balance + 100 WHERE account_id = 1")
spark.sql(f"DELETE FROM {table} WHERE balance < 100")

display(spark.sql(f"DESCRIBE HISTORY {table}").select("version", "timestamp", "operation", "operationParameters", "operationMetrics"))

## 3. Time travel in one line

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads version 0 and the current version of the table.<br><br>
<b>Why it matters</b><br>Because old files aren't deleted immediately, any retained version can be queried. This is a preview of lesson 08.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Version 0 has 4 rows with Amal at 1200.0. The current version has 4 rows (Priya added, Sara deleted) with Amal at 1300.0.
</div>

In [0]:
spark.sql(f"SELECT * FROM {table} VERSION AS OF 0 ORDER BY account_id").show()
spark.table(table).orderBy("account_id").show()

## 4. Look at the files: a path-based Delta table

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the same data as a Delta table at a volume path, then lists the data files and the <code>_delta_log</code> folder.<br><br>
<b>Why it matters</b><br>Managed tables hide their storage location, which is a good thing in production. Writing to a path lets you <b>see</b> the structure: Parquet data files next to a <code>_delta_log</code> folder of numbered JSON commits.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Unity Catalog recommends registered tables for tabular data, and volumes for files. Writing Delta to a volume path is used here only to inspect the format.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Some <code>part-...parquet</code> files and a <code>_delta_log</code> folder containing <code>00000000000000000000.json</code>.
</div>

In [0]:
import os

path = f"{BASE}/accounts_delta"
accounts.write.format("delta").mode("overwrite").save(path)

print("data files:", sorted(f for f in os.listdir(path) if f.endswith(".parquet")))
print("log files :", sorted(f for f in os.listdir(f"{path}/_delta_log") if not f.startswith(".")))

## 5. Delta vs plain Parquet: what an overwrite does

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Overwrites a Delta path and a Parquet path, then checks whether the previous data is still available.<br><br>
<b>Why it matters</b><br>A Parquet overwrite <b>deletes</b> the old files, so the previous data is gone (and a failure mid-write leaves a broken folder). A Delta overwrite only <b>commits</b> a new version: the old files stay referenced by the old version until <code>VACUUM</code>, so you can read or restore them.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Delta: version 0 is still readable with 4 rows after the overwrite. Parquet: only the new data exists.
</div>

In [0]:
parquet_path = f"{BASE}/accounts_parquet"
accounts.write.mode("overwrite").parquet(parquet_path)

new_data = spark.createDataFrame([(9, "Ravi", 10.0)], accounts.schema)
new_data.write.format("delta").mode("overwrite").save(path)
new_data.write.mode("overwrite").parquet(parquet_path)

print("Delta, current rows  :", spark.read.format("delta").load(path).count())
print("Delta, version 0 rows:", spark.read.format("delta").option("versionAsOf", 0).load(path).count())
print("Parquet, rows        :", spark.read.parquet(parquet_path).count(), "(the old data is gone)")

## 6. Delta in SQL and in Python

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Queries the same table with SQL, <code>spark.table</code>, the <code>DeltaTable</code> API, and by path.<br><br>
<b>Why it matters</b><br>You'll see all four styles. The <code>DeltaTable</code> class (from <code>delta.tables</code>) gives Python methods for operations such as <code>update</code>, <code>delete</code>, <code>merge</code>, <code>history</code> and <code>optimize</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The same 4 rows four times, and the last 2 history entries from the <code>DeltaTable</code> API.
</div>

In [0]:
from delta.tables import DeltaTable

spark.sql(f"SELECT count(*) AS rows FROM {table}").show()
print("spark.table :", spark.table(table).count())

dt = DeltaTable.forName(spark, table)
print("DeltaTable  :", dt.toDF().count())
dt.history(2).select("version", "operation").show()

print("by path     :", spark.read.format("delta").load(path).count())

## Under the hood

```
WRITE (commit N)                                    READ (latest)
1. write new Parquet files (not visible yet)        1. find the latest checkpoint + later JSON commits
2. try to create _delta_log/N.json atomically       2. replay add/remove actions -> list of live files
   - success: version N exists, files visible       3. use per-file min/max stats to skip files
   - someone else created N first: check conflicts  4. read only those Parquet files
     and retry as N+1, or fail
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> every write, `UPDATE`, `DELETE` or `MERGE` creates exactly one new version, or none if it fails.

The **atomic step** is creating the next log file: storage guarantees only one writer can create `00000000000000000005.json`. Lesson 03 opens these files and reads them.

In [0]:
spark.table(table).filter("balance > 1000").explain()

In [0]:
# Optional cleanup: remove this lesson's files (the accounts table is reused in later lessons).
dbutils.fs.rm(BASE, True)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: reading a Delta folder with <code>spark.read.parquet</code></b><br>
You get every Parquet file, including ones removed by later commits, so duplicates and deleted rows reappear. Always read with <code>format("delta")</code> or by table name.<br><br>
<b>⛔ Problem: deleting files inside a Delta table folder by hand</b><br>The log still references them, and reads fail with <code>FileNotFoundException</code>. Use <code>DELETE</code>, <code>VACUUM</code> and <code>RESTORE</code>.<br><br>
**⛔ Problem: <code>DROP TABLE</code> deleted the data**<br>That's expected for managed tables. External tables keep their files.<br><br>
**⛔ Problem: <code>DELTA_MISSING_TRANSACTION_LOG</code>**<br>The path isn't a Delta table (no <code>_delta_log</code>), or you pointed at a subfolder.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is Delta Lake?</b><br>
An open-source storage layer that stores data as Parquet files plus a transaction log. The log brings ACID transactions, schema enforcement and evolution, <code>UPDATE</code>/<code>DELETE</code>/<code>MERGE</code>, time travel, and file-level statistics to data lakes. It's the default table format on Databricks.<br><br>

<b>🎤 2. How does Delta provide ACID guarantees?</b><br>
Every change writes new data files first, then commits by atomically creating the next numbered JSON file in <code>_delta_log</code>, which lists the files added and removed. Readers only see files from committed versions, so writes are atomic and isolated. Concurrent writers use optimistic concurrency control: if two try to create the same version, one retries after checking for conflicts. Committed versions are durable in cloud storage.<br><br>

<b>🎤 3. What is the difference between a managed and an external table?</b><br>
For a managed table, Unity Catalog manages both metadata and storage, and dropping the table deletes the data. An external table's data lives at a path you specify, and dropping it removes only the metadata. Managed tables are recommended, because they get automatic optimizations.<br><br>

<b>🎤 4. What happens to the old data when you overwrite a Delta table?</b><br>
The new version references the new files and marks the old ones as removed in the log, but the files stay on storage until <code>VACUUM</code> deletes them after the retention period. So old versions can still be queried or restored.<br><br>

<b>🎤 5. Why can't you read a Delta table with <code>spark.read.parquet</code>?</b><br>
You'd read every Parquet file in the folder, including files that later commits removed. Only the transaction log knows which files are part of the current version.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. What does the <code>_delta_log</code> directory of a Delta table contain?</b><br>
A. Copies of all data files<br>
B. JSON commit files (and periodic Parquet checkpoints) recording each transaction's added and removed files and metadata<br>
C. Only the table schema<br>
D. Spark executor logs<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A data engineer runs <code>DROP TABLE sales</code> on a managed Delta table in Unity Catalog. What happens to the data files?</b><br>
A. They remain in place indefinitely<br>
B. They're deleted along with the metadata (with a short window to <code>UNDROP</code>)<br>
C. They're moved to an external location<br>
D. They're converted to Parquet<br>
<details><summary><b>Show answer</b></summary><b>B.</b> For an external table, the files would remain.</details><br><br>

<b>Q3. Which file format stores the data of a Delta table?</b><br>
A. Avro<br>
B. JSON<br>
C. Parquet<br>
D. ORC<br>
<details><summary><b>Show answer</b></summary><b>C.</b> JSON (and Parquet checkpoints) are used only for the transaction log.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a managed table <code>products</code> with 3 rows, then append 2 rows, then update one price</li><li>Show the history, and read the table as it was before the update</li><li>Write the same data as Delta to a volume path, and list its <code>_delta_log</code> files after two appends</li><li>In two sentences, answer "How does Delta provide ACID guarantees?" out loud</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Create, append, update
products = f"{DB}.products"
spark.createDataFrame([(1, "Laptop", 900.0), (2, "Mouse", 20.0), (3, "Desk", 250.0)], "id INT, name STRING, price DOUBLE") \
    .write.mode("overwrite").saveAsTable(products)
spark.createDataFrame([(4, "Lamp", 35.0), (5, "Chair", 120.0)], "id INT, name STRING, price DOUBLE") \
    .write.mode("append").saveAsTable(products)
spark.sql(f"UPDATE {products} SET price = 850.0 WHERE id = 1")

# 2. History and the version before the update (version 1)
spark.sql(f"DESCRIBE HISTORY {products}").select("version", "operation").show()
spark.sql(f"SELECT * FROM {products} VERSION AS OF 1 WHERE id = 1").show()

# 3. Path-based table and its log
ppath = f"{BASE}/products_delta"
spark.table(products).write.format("delta").mode("overwrite").save(ppath)
for i in range(2):
    spark.createDataFrame([(10 + i, f"Extra {i}", 1.0)], "id INT, name STRING, price DOUBLE").write.format("delta").mode("append").save(ppath)
print(sorted(f for f in os.listdir(f"{ppath}/_delta_log") if f.endswith(".json")))

# 4. Changes write new files, then commit atomically by creating the next log file; readers only see committed files.
#    Concurrent writers use optimistic concurrency control, so conflicting commits retry or fail instead of corrupting data.
spark.sql(f"DROP TABLE IF EXISTS {products}")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Delta = Parquet data files + <code>_delta_log</code> transaction log (JSON commits + Parquet checkpoints)</li><li>Every successful change is one atomic commit = one version</li><li>Adds ACID, schema enforcement, DML, time travel, file statistics, and streaming support to data lakes</li><li>Delta is the default table format on Databricks</li><li>Managed tables: UC manages data, and <code>DROP</code> deletes it. External: your path, and <code>DROP</code> keeps the files</li><li>Never read a Delta folder as plain Parquet, and never delete its files by hand</li></ul>
</div>

| Task | Code |
|---|---|
| Create (managed) | `df.write.saveAsTable("cat.sch.t")` |
| Create (path) | `df.write.format("delta").save(path)` |
| Details | `DESCRIBE DETAIL t` |
| History | `DESCRIBE HISTORY t`, `DeltaTable.forName(spark, "t").history()` |
| Old version | `SELECT * FROM t VERSION AS OF 3` |
| Python API | `from delta.tables import DeltaTable` |

## Git commit

```
git add 03_delta_lake/01_delta_fundamentals.ipynb
git commit -m "add 03_01 delta fundamentals notebook"
```